In [41]:
import matplotlib
import numpy as np
import pandas as pd
from narwhals import Int64
import seaborn as sns
from scipy.special.cython_special import stdtrit
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
from sklearn.base import clone
from scipy import special

In [ ]:
train_df = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index_col=0)

# Preprocessing before pipeline and modeling
## We drop unnecessary columns, define categorical and discrete/numerical columns for imputation, and convert categorical variables into strings

In [ ]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")
# Defining categorical columns
cat_cols = ["SubjectRace", "Zip", "District", "StopDescription"]
# Defining discrete/numerical columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "SubjectHasPhotoID", "hour", "night", "isDriver"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "hour"]
# Binary columns
bin_cols = ["SubjectGender", "SubjectHasPhotoID", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)

In [ ]:
target = "search_occurred"
X_train = train_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_train = train_df[target]

# Defining classes to build pipeline
## CatBinner groups sparse labels in categorical variables in an Other category. RoundImputed is used after imputation, to ensure imputed numerical values are discrete

In [ ]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class OrdEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, inner):
        self.cat_cols = cat_cols
        self.inner = inner

    def _encode(self, X):
        X_enc = X.copy()
        X_enc[self.cat_cols] = self.encoder_.transform(X[self.cat_cols])
        return X_enc

    def _decode(self, X_imp):
        X_dec = X_imp.copy()
        X_dec[self.cat_cols] = self.encoder_.inverse_transform(X_imp[self.cat_cols])
        return X_dec

    def fit_transform(self, X, y=None):
        self.encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
        self.encoder_.fit(X[self.cat_cols])
        self.inner_ = clone(self.inner)
        return self._decode(self.inner_.fit_transform(self._encode(X), y))

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        return self._decode(self.inner_.transform(self._encode(X)))

    def set_output(self, *, transform=None):
        return self

# Building pipeline
## Define a function to construct pipeline given model, and discrete and categorical columns

In [ ]:
def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10), skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    impute = Pipeline([
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
    ])

    impute.set_output(transform="pandas")

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("imputer", OrdEncoder(cat_cols, impute)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    return pipeline

## Defining ECE scorer

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=20):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    ece = 0.0
    for i in range(n_bins):
        bin_mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        if np.any(bin_mask):
            bin_accuracy = np.mean(y_true[bin_mask])
            bin_confidence = bin_centers[i]
            ece += np.abs(bin_confidence - bin_accuracy) * np.sum(bin_mask)
    ece /= y_true.size
    return ece

ece = make_scorer(expected_calibration_error, greater_is_better=False, response_method="predict_proba", n_bins = 20)

# Modeling
## Logistic regression

In [ ]:
# USE OPTIMAL PARAMETERS FOUND IN TUNING
lr = LogisticRegression(l1_ratio=1, C = 0.012742749857031334, max_iter = 1000, random_state=10, solver="saga")

pipeline = build_pipeline(lr, cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=10)

lr_cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring={"average_precision":"average_precision", "roc_auc":"roc_auc", "neg_brier_score":"neg_brier_score", "neg_ece":ece}, return_train_score=True, error_score="raise", verbose=2)

## Decision Tree

In [ ]:
dt = DecisionTreeClassifier(criterion = "entropy", max_depth = 9, min_samples_leaf = 31, min_samples_split = 29, random_state=10)

pipeline = build_pipeline(dt, cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=10)

dt_cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring={"average_precision":"average_precision", "roc_auc":"roc_auc", "neg_brier_score":"neg_brier_score", "neg_ece":ece}, return_train_score=True, error_score="raise", verbose=2)

## Random Forest

In [ ]:
rf = RandomForestClassifier(criterion="gini", max_depth=None, min_samples_split=13, min_samples_leaf=11, max_features="log2", random_state=10)

pipeline = build_pipeline(rf, cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=10)

rf_cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring={"average_precision":"average_precision", "roc_auc":"roc_auc", "neg_brier_score":"neg_brier_score", "neg_ece":ece}, return_train_score=True, error_score="raise", verbose=2)

# Evaluating metrics

In [ ]:
# Saving results
lr_scores = pd.DataFrame(lr_cross_val_scores)
dt_scores = pd.DataFrame(dt_cross_val_scores)
rf_scores = pd.DataFrame(rf_cross_val_scores)

# Making brier and ece nonnegative
lr_scores["test_brier_score"], lr_scores["train_brier_score"], lr_scores["test_ece"], lr_scores["train_ece"] = -lr_scores["test_neg_brier_score"], -lr_scores["train_neg_brier_score"], -lr_scores["test_neg_ece"], -lr_scores["train_neg_ece"]

dt_scores["test_brier_score"], dt_scores["train_brier_score"], dt_scores["test_ece"], dt_scores["train_ece"] = -dt_scores["test_neg_brier_score"], -dt_scores["train_neg_brier_score"], -dt_scores["test_neg_ece"], -dt_scores["train_neg_ece"]

rf_scores["test_brier_score"], rf_scores["train_brier_score"], rf_scores["test_ece"], rf_scores["train_ece"] = -rf_scores["test_neg_brier_score"], -rf_scores["train_neg_brier_score"], -rf_scores["test_neg_ece"], -rf_scores["train_neg_ece"]

# Creating standard deviation and error columns
t_alpha = special.stdtrit(len(lr_scores)-1, 0.975)
summary_lr = (lr_scores.agg(["mean", "std"]).T.reset_index())
summary_lr.columns = ["metric", "lr_mean", "lr_std"]
summary_lr["lr_se"] = summary_lr["lr_std"]/np.sqrt(len(lr_scores))
summary_lr["lr_lower_95_ci"] = summary_lr["lr_mean"] - t_alpha*summary_lr["lr_se"]
summary_lr["lr_upper_95_ci"] = summary_lr["lr_mean"] + t_alpha*summary_lr["lr_se"]

summary_dt = (dt_scores.agg(["mean", "std"]).T.reset_index())
summary_dt.columns = ["metric", "dt_mean", "dt_std"]
summary_dt["dt_se"] = summary_dt["dt_std"]/np.sqrt(len(dt_scores))
summary_dt["dt_lower_95_ci"] = summary_dt["dt_mean"] - t_alpha*summary_dt["dt_se"]
summary_dt["dt_upper_95_ci"] = summary_dt["dt_mean"] + t_alpha*summary_dt["dt_se"]

summary_rf = (rf_scores.agg(["mean", "std"]).T.reset_index())
summary_rf.columns = ["metric", "rf_mean", "rf_std"]
summary_rf["rf_se"] = summary_rf["rf_std"]/np.sqrt(len(rf_scores))
summary_rf["rf_lower_95_ci"] = summary_rf["rf_mean"] - t_alpha*summary_rf["rf_se"]
summary_rf["rf_upper_95_ci"] = summary_rf["rf_mean"] + t_alpha*summary_rf["rf_se"]

In [ ]:
# Combining results into dataframe
score_summary = pd.concat([summary_lr, summary_dt, summary_rf], axis=1)
# Drop duplicate metric columns
score_summary = score_summary.T.drop_duplicates().T
score_summary

In [ ]:
# Saving results
summary_lr.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\lr_cross_val_scores.csv", index=False)
summary_dt.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\dt_cross_val_scores.csv", index=False)
summary_rf.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\rf_cross_val_scores.csv", index=False)
score_summary.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\model_cross_val_scores.csv", index=False)

## Plotting cross validation scores

In [ ]:
lr_scores["Model"] = "Logistic Regression"
dt_scores["Model"] = "Decision Tree"
rf_scores["Model"] = "Random Forest"
scores = pd.concat([lr_scores, dt_scores, rf_scores])
scores

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(15,10))

sns.boxplot(scores, x = "Model", y = "test_average_precision", order=["Logistic Regression", "Decision Tree", "Random Forest"], ax = axs[0,0])
axs[0,0].set_ylabel("Validation PR-AUC/Average Precision", fontsize=12)
axs[0,0].set_title("PR-AUC", fontsize=14)

sns.boxplot(scores, x = "Model", y = "test_brier_score", order=["Logistic Regression", "Decision Tree", "Random Forest"], ax = axs[0,1])
axs[0,1].set_ylabel("Validation Brier Score", fontsize=12)
axs[0,1].set_title("Brier Score", fontsize=14)

sns.boxplot(scores, x = "Model", y = "test_roc_auc", order=["Logistic Regression", "Decision Tree", "Random Forest"], ax = axs[1,0])
axs[1,0].set_ylabel("Validation ROC-AUC", fontsize=12)
axs[1,0].set_title("ROC-AUC", fontsize=14)

sns.boxplot(scores, x = "Model", y = "test_ece", order=["Logistic Regression", "Decision Tree", "Random Forest"], ax = axs[1,1])
axs[1,1].set_ylabel("Validation Expected Calibration Error", fontsize=12)
axs[1,1].set_title("Expected Calibration Error", fontsize=14)

plt.suptitle("10-Fold CV Performance Metrics of Models", fontsize=30)
plt.savefig(r"C:\Users\gelet\PycharmProjects\stop-and-search\plots\cv_models_boxplot.png", bbox_inches="tight")
plt.show()